 # Bitcoin testing

In [1]:
import torch

model = torch.load("BTC_final_model.pth", weights_only = False)
model.eval()

BiLSTM(
  (lstm): LSTM(29, 64, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [2]:
import joblib

x_scaler = joblib.load("btc_x_scaler.pkl")
y_scaler = joblib.load("btc_y_scaler.pkl")

In [3]:
import pandas as pd

bit = pd.read_csv("BTC-USD_last_3_year.csv")

bit = bit.iloc[2:].reset_index(drop=True)
bit.columns.values[0] = "Date"

In [4]:
bit["Date"] = pd.to_datetime(bit["Date"])

cols = ["Close", "High", "Low", "Open", "Volume"]
bit[cols] = bit[cols].apply(pd.to_numeric)

bit = bit.dropna().reset_index(drop=True)

In [5]:
bit.dtypes

Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

In [6]:
from feature_engineering import calculate_features

bit = calculate_features(bit)

In [7]:
bit = bit.dropna().reset_index(drop=True)

In [8]:
X_bit = bit.drop(columns=["Close"]).values

In [9]:
X_scaled = x_scaler.transform(X_bit)

In [10]:
import numpy as np

def create_sequences(X, time_steps=30):
    X_seq = []
    for i in range(len(X) - time_steps):
        X_seq.append(X[i:i+time_steps])
    return np.array(X_seq)

X_seq = create_sequences(X_scaled)

In [11]:
preds = model(torch.tensor(X_seq, dtype=torch.float32)).detach().numpy()

In [12]:
preds_actual = y_scaler.inverse_transform(preds.reshape(-1,1))

In [13]:
y_actual = bit["Close"].shift(-1).dropna().values[30:].reshape(-1,1)

# align predictions length
preds_actual = preds_actual[:len(y_actual)]

In [14]:
import pandas as pd

df_results = pd.DataFrame({
    "Actual": y_actual.flatten(),
    "Predicted": preds_actual.flatten()
})

df_results.head()

,Actual,Predicted
0,26861.707031,26619.964844
1,27159.652344,27049.207031
2,28519.466797,27562.277344
3,28415.748047,27985.794922
4,28328.341797,28298.468750


In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_actual, preds_actual)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_actual, preds_actual)
r2 = r2_score(y_actual, preds_actual)
mape = np.mean(np.abs((y_actual - preds_actual) / y_actual)) * 100

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

MSE: 14098784.23
RMSE: 3754.83
MAE: 2777.22
R2 Score: 0.9652
MAPE: 3.87%


# Binance Testing

In [16]:
import torch

model = torch.load("BNB_final_model.pth", weights_only = False)
model.eval()

BiLSTM(
  (lstm): LSTM(29, 39, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=78, out_features=1, bias=True)
)

In [17]:
import joblib

x_scaler = joblib.load("bnb_x_scaler.pkl")
y_scaler = joblib.load("bnb_y_scaler.pkl")

In [18]:
import pandas as pd

bnb = pd.read_csv("BNB-USD_last_3_year.csv")

bnb = bnb.iloc[2:].reset_index(drop=True)
bnb.columns.values[0] = "Date"

In [19]:
bnb["Date"] = pd.to_datetime(bnb["Date"])

cols = ["Close", "High", "Low", "Open", "Volume"]
bnb[cols] = bnb[cols].apply(pd.to_numeric)

bnb = bnb.dropna().reset_index(drop=True)

In [20]:
bnb.dtypes

Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

In [21]:
from feature_engineering import calculate_features

bnb = calculate_features(bnb)

In [22]:
bnb = bnb.dropna().reset_index(drop=True)

In [23]:
X_bnb = bnb.drop(columns=["Close"]).values

In [24]:
X_scaled = x_scaler.transform(X_bnb)

In [25]:
import numpy as np

def create_sequences(X, time_steps=30):
    X_seq = []
    for i in range(len(X) - time_steps):
        X_seq.append(X[i:i+time_steps])
    return np.array(X_seq)

X_seq = create_sequences(X_scaled)

In [26]:
preds = model(torch.tensor(X_seq, dtype=torch.float32)).detach().numpy()

In [27]:
preds_actual = y_scaler.inverse_transform(preds.reshape(-1,1))

In [28]:
y_actual = bnb["Close"].shift(-1).dropna().values[30:].reshape(-1,1)

# align predictions length
preds_actual = preds_actual[:len(y_actual)]

In [29]:
import pandas as pd

df_results = pd.DataFrame({
    "Actual": y_actual.flatten(),
    "Predicted": preds_actual.flatten()
})

df_results.head()

,Actual,Predicted
0,206.601898,218.897751
1,209.742508,220.595474
2,214.823959,221.598679
3,211.643234,222.348145
4,210.501038,225.129303


In [30]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_actual, preds_actual)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_actual, preds_actual)
r2 = r2_score(y_actual, preds_actual)
mape = np.mean(np.abs((y_actual - preds_actual) / y_actual)) * 100

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

MSE: 1646.61
RMSE: 40.58
MAE: 29.44
R2 Score: 0.9298
MAPE: 5.11%


# Ethereum testing

In [31]:
import torch

model = torch.load("ETH_final_model.pth", weights_only = False)
model.eval()

BiLSTM(
  (lstm): LSTM(29, 64, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [32]:
import joblib

x_scaler = joblib.load("eth_x_scaler.pkl")
y_scaler = joblib.load("eth_y_scaler.pkl")

In [33]:
import pandas as pd

eth = pd.read_csv("ETH-USD_last_3_year.csv")

eth = eth.iloc[2:].reset_index(drop=True)
eth.columns.values[0] = "Date"

In [34]:
eth["Date"] = pd.to_datetime(eth["Date"])

cols = ["Close", "High", "Low", "Open", "Volume"]
eth[cols] = eth[cols].apply(pd.to_numeric)

eth = eth.dropna().reset_index(drop=True)

In [35]:
eth.dtypes

Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

In [36]:
from feature_engineering import calculate_features

eth = calculate_features(eth)

In [37]:
eth = eth.dropna().reset_index(drop=True)

In [38]:
X_eth = eth.drop(columns=["Close"]).values

In [39]:
X_scaled = x_scaler.transform(X_eth)

In [40]:
import numpy as np

def create_sequences(X, time_steps=30):
    X_seq = []
    for i in range(len(X) - time_steps):
        X_seq.append(X[i:i+time_steps])
    return np.array(X_seq)

X_seq = create_sequences(X_scaled)

In [41]:
preds = model(torch.tensor(X_seq, dtype=torch.float32)).detach().numpy()

In [42]:
preds_actual = y_scaler.inverse_transform(preds.reshape(-1,1))

In [43]:
y_actual = eth["Close"].shift(-1).dropna().values[30:].reshape(-1,1)

# align predictions length
preds_actual = preds_actual[:len(y_actual)]

In [44]:
import pandas as pd

df_results = pd.DataFrame({
    "Actual": y_actual.flatten(),
    "Predicted": preds_actual.flatten()
})

df_results.head()

,Actual,Predicted
0,1555.256836,1567.062622
1,1558.069824,1588.213989
2,1600.534302,1635.264648
3,1565.439575,1661.457275
4,1563.749878,1662.251709


In [45]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_actual, preds_actual)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_actual, preds_actual)
r2 = r2_score(y_actual, preds_actual)
mape = np.mean(np.abs((y_actual - preds_actual) / y_actual)) * 100

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

MSE: 171967.65
RMSE: 414.69
MAE: 344.10
R2 Score: 0.5213
MAPE: 11.38%


# Ripple testing

In [46]:
import torch

model = torch.load("XRP_final_model.pth", weights_only = False)
model.eval()

BiLSTM(
  (lstm): LSTM(29, 64, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [47]:
import joblib

x_scaler = joblib.load("xrp_x_scaler.pkl")
y_scaler = joblib.load("xrp_y_scaler.pkl")

In [48]:
import pandas as pd

xrp = pd.read_csv("XRP-USD_last_3_year.csv")

xrp = xrp.iloc[2:].reset_index(drop=True)
xrp.columns.values[0] = "Date"

In [49]:
xrp["Date"] = pd.to_datetime(xrp["Date"])

cols = ["Close", "High", "Low", "Open", "Volume"]
xrp[cols] = xrp[cols].apply(pd.to_numeric)

xrp = xrp.dropna().reset_index(drop=True)

In [50]:
xrp.dtypes

Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

In [51]:
from feature_engineering import calculate_features

xrp = calculate_features(xrp)

In [52]:
xrp = xrp.dropna().reset_index(drop=True)

In [53]:
X_xrp = xrp.drop(columns=["Close"]).values

In [54]:
X_scaled = x_scaler.transform(X_xrp)

In [55]:
import numpy as np

def create_sequences(X, time_steps=30):
    X_seq = []
    for i in range(len(X) - time_steps):
        X_seq.append(X[i:i+time_steps])
    return np.array(X_seq)

X_seq = create_sequences(X_scaled)

In [56]:
preds = model(torch.tensor(X_seq, dtype=torch.float32)).detach().numpy()

In [57]:
preds_actual = y_scaler.inverse_transform(preds.reshape(-1,1))

In [58]:
y_actual = xrp["Close"].shift(-1).dropna().values[30:].reshape(-1,1)

# align predictions length
preds_actual = preds_actual[:len(y_actual)]

In [59]:
import pandas as pd

df_results = pd.DataFrame({
    "Actual": y_actual.flatten(),
    "Predicted": preds_actual.flatten()
})

df_results.head()

,Actual,Predicted
0,0.486775,0.461824
1,0.487846,0.462269
2,0.497977,0.466863
3,0.491694,0.476478
4,0.488319,0.490458


In [60]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_actual, preds_actual)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_actual, preds_actual)
r2 = r2_score(y_actual, preds_actual)
mape = np.mean(np.abs((y_actual - preds_actual) / y_actual)) * 100

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

MSE: 0.18
RMSE: 0.42
MAE: 0.31
R2 Score: 0.7282
MAPE: 32.64%


# Solana testing

In [16]:
import torch

model = torch.load("SOL_final_model.pth", weights_only = False)
model.eval()

BiLSTM(
  (lstm): LSTM(29, 64, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [17]:
import joblib

x_scaler = joblib.load("sol_x_scaler.pkl")
y_scaler = joblib.load("sol_y_scaler.pkl")

In [18]:
import pandas as pd

sol = pd.read_csv("SOL-USD_last_3_year.csv")

sol = sol.iloc[2:].reset_index(drop=True)
sol.columns.values[0] = "Date"

In [19]:
sol["Date"] = pd.to_datetime(sol["Date"])

cols = ["Close", "High", "Low", "Open", "Volume"]
sol[cols] = sol[cols].apply(pd.to_numeric)

sol = sol.dropna().reset_index(drop=True)

In [20]:
sol.dtypes

Date      datetime64[ns]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

In [21]:
from feature_engineering import calculate_features

sol = calculate_features(sol)

In [22]:
sol = sol.dropna().reset_index(drop=True)

In [23]:
X_sol = sol.drop(columns=["Close"]).values

In [24]:
X_scaled = x_scaler.transform(X_sol)

In [25]:
import numpy as np

def create_sequences(X, time_steps=30):
    X_seq = []
    for i in range(len(X) - time_steps):
        X_seq.append(X[i:i+time_steps])
    return np.array(X_seq)

X_seq = create_sequences(X_scaled)

In [26]:
preds = model(torch.tensor(X_seq, dtype=torch.float32)).detach().numpy()

In [27]:
preds_actual = y_scaler.inverse_transform(preds.reshape(-1,1))

In [28]:
y_actual = sol["Close"].shift(-1).dropna().values[30:].reshape(-1,1)

# align predictions length
preds_actual = preds_actual[:len(y_actual)]

In [29]:
import pandas as pd

df_results = pd.DataFrame({
    "Actual": y_actual.flatten(),
    "Predicted": preds_actual.flatten()
})

df_results.head()

,Actual,Predicted
0,22.011719,17.427307
1,21.922028,18.767139
2,23.982958,20.397886
3,23.959318,20.934017
4,23.432138,20.004948


In [30]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_actual, preds_actual)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_actual, preds_actual)
r2 = r2_score(y_actual, preds_actual)
mape = np.mean(np.abs((y_actual - preds_actual) / y_actual)) * 100

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

MSE: 409.11
RMSE: 20.23
MAE: 17.02
R2 Score: 0.8512
MAPE: 11.87%
